In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay

# Advanced Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Create artifacts directory
os.makedirs('../artifacts', exist_ok=True)
sns.set_theme(style="whitegrid")
print("Environment ready!")

In [ ]:
# Paths relative to /notebooks folder
train_path = '../data/train.csv'
test_path = '../data/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df.drop('loan_status', axis=1)
y_train = train_df['loan_status']
X_test = test_df.drop('loan_status', axis=1)
y_test = test_df['loan_status']

print(f"Data Loaded. Training set distribution:\n{y_train.value_counts()}")

In [ ]:
# Cell 3: Preprocessing Pipeline (Warning-Free Version)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Select numeric and categorical features
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
# FIXED: Explicitly added 'string' to silence the Pandas4Warning
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing pipeline ready (and warning-free)!")

In [ ]:
print("Training Random Forest...")
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', RandomForestClassifier(random_state=42))])

rf_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, None]
}

rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
print(f"Best Random Forest AUC: {rf_grid.best_score_:.4f}")

In [ ]:
print("Training Logistic Regression...")
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', LogisticRegression(random_state=42, max_iter=1000))])

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
lr_param_grid = {'classifier__C': [0.1, 1, 10]}

lr_grid = GridSearchCV(lr_pipeline, lr_param_grid, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
lr_grid.fit(X_train, y_train)
best_lr = lr_grid.best_estimator_
print(f"Best Logistic Regression AUC: {lr_grid.best_score_:.4f}")

In [ ]:
print("Training the Voting Classifier (Ensemble)...")
gb_model = GradientBoostingClassifier(random_state=42)
gb_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', gb_model)])
gb_pipeline.fit(X_train, y_train)

voting_clf = VotingClassifier(
    estimators=[('gb', gb_pipeline.named_steps['classifier']), 
                ('rf', best_rf.named_steps['classifier'])],
    voting='soft'
)

voting_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', voting_clf)])
voting_pipeline.fit(X_train, y_train)

y_pred_proba = voting_pipeline.predict_proba(X_test)[:, 1]
final_auc = roc_auc_score(y_test, y_pred_proba)
print(f"Voting Classifier Test AUC: {final_auc:.4f}")

In [ ]:
# Cell 5.5: Train Model 3 (Gradient Boosting)
from sklearn.ensemble import GradientBoostingClassifier

print("Training Gradient Boosting Classifier with Grid Search...")
gb_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', GradientBoostingClassifier(random_state=42))])

# Testing different hyperparameters
gb_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__max_depth': [3, 5]
}

gb_grid = GridSearchCV(gb_pipeline, gb_param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
gb_grid.fit(X_train, y_train)

best_gb = gb_grid.best_estimator_
print(f"Best Gradient Boosting AUC: {gb_grid.best_score_:.4f}")

In [ ]:
# Cell 5.6: Train Model 4 (XGBoost)
from xgboost import XGBClassifier

print("Training XGBoost Classifier with Grid Search...")
xgb_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))])

# Testing different hyperparameters
xgb_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__max_depth': [3, 5],
    'classifier__subsample': [0.8, 1.0] # XGBoost specific tuning
}

xgb_grid = GridSearchCV(xgb_pipeline, xgb_param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
xgb_grid.fit(X_train, y_train)

best_xgb = xgb_grid.best_estimator_
print(f"Best XGBoost AUC: {xgb_grid.best_score_:.4f}")

In [ ]:
# Cell 5.7: Train Model 5 (LightGBM)
from lightgbm import LGBMClassifier

print("Training LightGBM Classifier with Grid Search...")
lgbm_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', LGBMClassifier(random_state=42, verbose=-1))])

# LightGBM specific tuning (it uses num_leaves as its main complexity dial)
lgbm_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__num_leaves': [31, 50] 
}

lgbm_grid = GridSearchCV(lgbm_pipeline, lgbm_param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
lgbm_grid.fit(X_train, y_train)

best_lgbm = lgbm_grid.best_estimator_
print(f"Best LightGBM AUC: {lgbm_grid.best_score_:.4f}")

In [ ]:
# Cell 5.8: The Avengers
from sklearn.ensemble import VotingClassifier

print("Training the Voting Classifier (Combining the Top 3)...")

# We combine the actual models we already tuned (stripping them out of their pipelines temporarily)
voting_clf = VotingClassifier(
    estimators=[
        ('gb', best_gb.named_steps['classifier']),
        ('xgb', best_xgb.named_steps['classifier']),
        ('lgbm', best_lgbm.named_steps['classifier'])
    ],
    voting='soft' # 'soft' means it averages their exact probabilities, not just 0 or 1
)

# Put them back into our main pipeline to handle the missing values and encoding
voting_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                  ('classifier', voting_clf)])

# Train the super-model!
voting_pipeline.fit(X_train, y_train)

# Evaluate it on the test set to see if it breaks the 0.9463 record!
voting_proba = voting_pipeline.predict_proba(X_test)[:, 1]
voting_auc = roc_auc_score(y_test, voting_proba)

print(f"Voting Classifier Test AUC: {voting_auc:.4f}")

In [ ]:
# Cell 7: Generate Visual Artifacts

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import warnings
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

# Safely ignore the harmless LightGBM feature name warning
warnings.filterwarnings("ignore", category=UserWarning, message="X does not have valid feature names")

# Ensure artifacts directory exists
os.makedirs('../artifacts', exist_ok=True)

print("Generating and saving visualizations...\n")

# Set global plot style
sns.set_theme(style="whitegrid")

# ==========================================
# 0. Calculate Feature Importances

def format_label(col):
    mapping = {
        'person_age': 'Age', 'person_income': 'Annual Income',
        'person_emp_length': 'Employment Length', 'loan_amnt': 'Loan Amount',
        'loan_int_rate': 'Interest Rate', 'loan_percent_income': 'Loan-to-Income Ratio',
        'cb_person_cred_hist_length': 'Credit History'
    }
    return mapping.get(col, col.replace('_', ' ').title())

cat_encoder = best_gb.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_features_encoded = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = list(numeric_features) + list(cat_features_encoded)

importances = best_gb.named_steps['classifier'].feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': [format_label(f) for f in all_feature_names],
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# ==========================================
# 1. Feature Importance Bar Chart

plt.figure(figsize=(10, 6))
top_features = feature_importance_df.head(10)

# FIXED: Added hue='Feature' and legend=False to silence Seaborn warning
sns.barplot(data=top_features, x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)

plt.title('Top 10 Feature Importances (Gradient Boosting)', fontsize=14, pad=15)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig('../artifacts/feature_importance.png', dpi=300)
plt.close()
print("Saved feature_importance.png")

# ==========================================
# 2. Confusion Matrix

plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay.from_estimator(
    voting_pipeline, 
    X_test, 
    y_test, 
    display_labels=['Non-Default (0)', 'Default (1)'],
    cmap='Blues',
    colorbar=False
)
plt.title('Confusion Matrix (Voting Classifier)', fontsize=14, pad=15)
plt.grid(False) 
plt.tight_layout()
plt.savefig('../artifacts/confusion_matrix.png', dpi=300)
plt.close('all') 
print("Saved confusion_matrix.png")

# ==========================================
# 3. ROC Curve

plt.figure(figsize=(8, 6))

# FIXED: Changed from plot_kwargs to curve_kwargs
disp = RocCurveDisplay.from_estimator(
    voting_pipeline, 
    X_test, 
    y_test,
    name='Voting Classifier (GB + XGB + LGBM)',
    curve_kwargs={'color': 'darkorange'} 
)

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('../artifacts/roc_curve.png', dpi=300)
plt.close('all')
print("Saved roc_curve.png\n")